In [1]:
from transformers import pipeline, set_seed
import torch

/experiment_dir/self_exp/triton_learning/triton_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import GPT2LMHeadModel

In [4]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [25]:
model_state = model.state_dict()

for k in model_state.keys():
    print(k, model_state[k].shape)

transformer.wte.weight torch.Size([50257, 768])
transformer.wpe.weight torch.Size([1024, 768])
transformer.h.0.ln_1.weight torch.Size([768])
transformer.h.0.ln_1.bias torch.Size([768])
transformer.h.0.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.0.attn.c_attn.bias torch.Size([2304])
transformer.h.0.attn.c_proj.weight torch.Size([768, 768])
transformer.h.0.attn.c_proj.bias torch.Size([768])
transformer.h.0.ln_2.weight torch.Size([768])
transformer.h.0.ln_2.bias torch.Size([768])
transformer.h.0.mlp.c_fc.weight torch.Size([768, 3072])
transformer.h.0.mlp.c_fc.bias torch.Size([3072])
transformer.h.0.mlp.c_proj.weight torch.Size([3072, 768])
transformer.h.0.mlp.c_proj.bias torch.Size([768])
transformer.h.1.ln_1.weight torch.Size([768])
transformer.h.1.ln_1.bias torch.Size([768])
transformer.h.1.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.1.attn.c_attn.bias torch.Size([2304])
transformer.h.1.attn.c_proj.weight torch.Size([768, 768])
transformer.h.1.attn.c_proj.bias 

In [15]:
x = torch.tensor([[1,2, 8], [3, 4, 9], [5, 6, 10]])

In [23]:
q   = x.split(1, dim=1)

In [16]:
x.shape

torch.Size([3, 3])

In [24]:
q

(tensor([[1],
         [3],
         [5]]),
 tensor([[2],
         [4],
         [6]]),
 tensor([[ 8],
         [ 9],
         [10]]))

# Sakespere Data Exp

In [1]:
import torch

In [2]:
with open("input.txt", "r") as f:
    text = f.read()

text[:1000]




"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [3]:
#character encoding
char_set = sorted(list(set(text)))
print(len(char_set), len(text))

65 1115394


In [4]:
vocab_size = len(char_set)
idx2char = {i:ch for i, ch in enumerate(char_set)}
char2idx  = {ch:i for i, ch in enumerate(char_set)}

encode = lambda s: [char2idx[c] for c in s]
decode = lambda l: "".join([idx2char[i] for i in l])

# print(encode("hii there"))
# print(decode(encode("hii there")))

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

torch.Size([1115394]) torch.int64


In [5]:
train_data[:9]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [6]:
block_size = 8 # context window
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context}, target: {target}")

when input is tensor([18]), target: 47
when input is tensor([18, 47]), target: 56
when input is tensor([18, 47, 56]), target: 57
when input is tensor([18, 47, 56, 57]), target: 58
when input is tensor([18, 47, 56, 57, 58]), target: 1
when input is tensor([18, 47, 56, 57, 58,  1]), target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]), target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), target: 58


In [7]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# read_text

with open("input.txt", "r") as f:
    text = f.read()

# print(text[0:100])

#character encoding
char_set = sorted(list(set(text)))
print(len(char_set), len(text))

vocab_size = len(char_set)
idx2char = {i:ch for i, ch in enumerate(char_set)}
char2idx  = {ch:i for i, ch in enumerate(char_set)}

encode = lambda s: [char2idx[c] for c in s]
decode = lambda l: "".join([idx2char[i] for i in l])

# print(encode("hii there"))
# print(decode(encode("hii there")))

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')

# for b in range(batch_size): # batch dimension
#     for t in range(block_size): # time dimension
#         context = xb[b, :t+1]
#         target = yb[b,t]
#         print(f"when input is {context.tolist()} the target: {target}")


class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        #idx: B,T , targets: B,T

        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx: B,T
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            #focus on last time step as bigram model
            logits = logits[:, -1, :]  # becomes B, C
            probs = F.softmax(logits, dim=-1) 
            idx_next = torch.multinomial(probs, num_samples=1) # B,1
            idx = torch.cat((idx, idx_next), dim=1) # B,T+1
        return idx # B, T+max_new_tokens

m = BigramLanguageModel()
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)



65 1115394
torch.Size([1115394]) torch.int64
torch.Size([32, 65])
tensor(4.7192, grad_fn=<NllLossBackward0>)


In [8]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


HM:B
PmXeSA;KuWUH hINfg VXeT!TCJkTABHf!TdEs
;a!FM
;bdTPq.xf?xzq.Z dEc!;fHpHfkuSpel.oiCxdHmXgM?fBsSSj


In [9]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.6025450229644775


In [10]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


Fo o bastrgeo m;
Went he d py Fike st I d whe s pr,

Palead hamerd-benlar.
tr bl othatut, be:
Be hrs


# Mathenatical trick in Self-Attention

In [11]:
# how token will communicate with all previous token ( think in that direction)
# take average of all previous + current token one type of comm.


torch.manual_seed(1337)
B,T,C = 4, 8, 2 #( B, T,C)
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [12]:
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev, 0)

In [13]:
wei = torch.tril(torch.ones(T,T))
wei = wei/wei.sum(1, keepdim=True)

In [14]:
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [15]:
xbow2 = wei@x # (B,T,T) @ (B,T,C)

In [16]:
torch.allclose(xbow, xbow2)

False

In [17]:
xbow.shape

torch.Size([4, 8, 2])

In [59]:
xbow[1], xbow2[1]

(tensor([[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]]),
 tensor([[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]]))

In [18]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T)) # like affinity matrix, how much token i is connected to token j, weight aggregation
wei = wei.masked_fill(tril == 0, float('-inf'))   # to stop communication from future token to past token
wei = F.softmax(wei, dim=-1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [19]:
xbow3 = wei@x
xbow3[1], xbow[1]


(tensor([[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]]),
 tensor([[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]]))

In [21]:
torch.allclose(xbow2, xbow3)

True

In [23]:
# version 4
# one head of self attention    

B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16

query = nn.Linear(C, head_size, bias=False)
key = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

q = query(x)  # (B, T, head_size)
k = key(x)    # (B, T, head_size)
wei = q@k.transpose(-2, -1) # (B, T, T)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
output = wei @ v # (B, T, T) @ (B,T,head_size) -> (B, T, head_size)

output.shape


torch.Size([4, 8, 16])

In [26]:
# Normalizing the attention scores
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

wei = q@k.transpose(-2, -1) # (B, T, T)

k.var(), q.var()


(tensor(0.8976), tensor(1.0298))

In [27]:
wei.var()  # in order of batch size

tensor(15.3940)

In [28]:
wei = wei/head_size**0.5
wei.var()


tensor(0.9621)

In [29]:
torch.softmax(torch.tensor([-0.2, -0.1, 0.1, 0.2, 0.5]), dim=-1)

tensor([0.1437, 0.1588, 0.1939, 0.2143, 0.2893])

In [32]:
torch.softmax(torch.tensor([-0.2, -0.1, 0.1, 0.2, 0.5])*8, dim=-1)

# you can see that the softmax is sharpen towards large values.

tensor([0.0032, 0.0072, 0.0356, 0.0793, 0.8746])